In [3]:
!pip install -q \
numpy==2.0.2 \
torch torchvision torchaudio \
opencv-python-headless==4.12.0.88 \
Pillow>=10.0.0 \
albumentations>=1.3.1 \
PyYAML>=6.0 \
tqdm>=4.65.0 \
scikit-learn>=1.3.0 \
matplotlib>=3.7.0 \
pandas>=2.0.0 \
seaborn>=0.12.0 \
roboflow>=1.1.9 \
wandb>=0.16.0 \
transformers==5.2.0 \
accelerate==1.12.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [1]:
import subprocess
result = subprocess.run(
    ["pip", "list", "--format=columns"],
    capture_output=True, text=True
)
# Filter relevant packages
keywords = ["torch", "numpy", "transformers", "albumentations",
            "opencv", "roboflow", "Pillow", "tqdm", "accelerate"]
for line in result.stdout.split("\n"):
    if any(k.lower() in line.lower() for k in keywords):
        print(line)

accelerate                               1.12.0
albumentations                           2.0.8
numpy                                    2.0.2
opencv-contrib-python                    4.12.0.88
opencv-python                            4.12.0.88
opencv-python-headless                   4.12.0.88
pillow                                   11.3.0
pytorch-ignite                           0.5.3
pytorch-lightning                        2.6.1
sentence-transformers                    5.2.0
torch                                    2.9.0+cu126
torchao                                  0.10.0
torchaudio                               2.9.0+cu126
torchcodec                               0.9.0
torchdata                                0.11.0
torchinfo                                1.8.0
torchmetrics                             1.8.2
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.24.0+cu126
tqdm                     

In [2]:
!pip install -q roboflow PyYAML seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [3]:
import torch
import numpy as np
import transformers
import albumentations
import cv2
import yaml
import seaborn

print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("transformers:", transformers.__version__)
print("albumentations:", albumentations.__version__)
print("cuda:", torch.cuda.is_available())

torch: 2.9.0+cu126
numpy: 2.0.2
transformers: 5.2.0
albumentations: 2.0.8
cuda: True


In [ ]:
!pip install -q roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="")  # paste your key here
project = rf.workspace("objectdetect-pu6rn").project("drywall-join-detect")
version = project.version(1)
dataset1 = version.download("coco", location="/kaggle/working/data/taping")
print("Dataset 1 downloaded")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/data/taping in coco:: 100%|██████████| 1192/1192 [00:00<00:00, 7994.49it/s]

Dataset 1 downloaded


In [5]:
project2 = rf.workspace("darpans-workspace").project("cracks-3ii36-lijfw")
version2 = project2.version(1)
dataset2 = version2.download("coco", location="/kaggle/working/data/crack")
print("Dataset 2 downloaded")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/data/crack in coco:: 100%|██████████| 5376/5376 [00:00<00:00, 5480.28it/s]

Dataset 2 downloaded


In [7]:
import os

for category in ["taping", "crack"]:
    for split in ["train", "valid", "test"]:
        path = f"/kaggle/working/data/{category}/{split}"
        if os.path.exists(path):
            files = os.listdir(path)
            imgs  = [f for f in files if f.endswith(".jpg") or f.endswith(".png")]
            print(f"{category}/{split}: {len(imgs)} images, {len(files)} total files")
        else:
            print(f"{category}/{split}: NOT FOUND")

taping/train: 936 images, 937 total files
taping/valid: 250 images, 251 total files
taping/test: NOT FOUND
crack/train: 5164 images, 5165 total files
crack/valid: 201 images, 202 total files
crack/test: 4 images, 5 total files


In [8]:
import json
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
import shutil

def coco_to_masks(data_dir, category):
    for split in ["train", "valid", "test"]:
        split_dir = Path(data_dir) / category / split
        ann_file  = split_dir / "_annotations.coco.json"
        
        if not ann_file.exists():
            print(f"No annotation file: {split_dir}")
            continue

        mask_dir = split_dir / "masks"
        mask_dir.mkdir(exist_ok=True)

        with open(ann_file) as f:
            data = json.load(f)

        # image_id → filename + size
        img_info = {img["id"]: img for img in data["images"]}

        # image_id → list of segmentation polygons
        ann_index = {}
        for ann in data.get("annotations", []):
            iid = ann["image_id"]
            ann_index.setdefault(iid, [])
            for seg in ann.get("segmentation", []):
                ann_index[iid].append(seg)

        count = 0
        for iid, info in img_info.items():
            w, h   = info["width"], info["height"]
            stem   = Path(info["file_name"]).stem
            mask   = Image.new("L", (w, h), 0)
            draw   = ImageDraw.Draw(mask)

            for poly in ann_index.get(iid, []):
                coords = [(poly[i], poly[i+1]) for i in range(0, len(poly)-1, 2)]
                if len(coords) >= 3:
                    draw.polygon(coords, fill=255)

            mask.save(mask_dir / f"{stem}.png")
            count += 1

        print(f"{category}/{split}: {count} masks created")

coco_to_masks("/kaggle/working/data", "taping")
coco_to_masks("/kaggle/working/data", "crack")

taping/train: 936 masks created
taping/valid: 250 masks created
No annotation file: /kaggle/working/data/taping/test
crack/train: 5164 masks created
crack/valid: 201 masks created
crack/test: 4 masks created


In [12]:
for category in ["taping", "crack"]:
    for split in ["train", "valid", "test"]:
        mask_dir = Path(f"/kaggle/working/data/{category}/{split}/masks")
        if mask_dir.exists():
            print(f"{category}/{split}: {len(list(mask_dir.glob('*.png')))} masks")

taping/train: 936 masks
taping/valid: 250 masks
crack/train: 5164 masks
crack/valid: 201 masks
crack/test: 4 masks


In [16]:
import os, random, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoProcessor, CLIPSegForImageSegmentation

# Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE     = torch.device("cuda")
DATA_ROOT  = "/kaggle/working/data"
CKPT_DIR   = "/kaggle/working/models"
IMG_SIZE   = 352
BATCH_SIZE = 16
EPOCHS     = 30
LR         = 3e-5
os.makedirs(CKPT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("CUDA device:", torch.cuda.get_device_name(0))

Device: cuda
CUDA device: Tesla T4


In [17]:
PROMPT_POOLS = {
    "taping": [
        "segment taping area",
        "segment joint tape",
        "segment drywall seam",
        "segment drywall joint",
    ],
    "crack": [
        "segment crack",
        "segment wall crack",
        "find crack",
        "highlight crack",
    ],
}

CANONICAL = {
    "taping": "segment taping area",
    "crack":  "segment crack",
}

def get_transforms(split):
    if split == "train":
        return A.Compose([
            A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.4),
            A.ColorJitter(brightness=0.2, contrast=0.2,
                          saturation=0.1, hue=0.05, p=0.5),
            A.Normalize(mean=(0.485,0.456,0.406),
                        std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(height=IMG_SIZE, width=IMG_SIZE),
            A.Normalize(mean=(0.485,0.456,0.406),
                        std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ])

class DrywallDataset(Dataset):
    def __init__(self, root, category, split, processor, augment_prompts=True):
        # map valid → val folder name
        self.split_dir  = Path(root) / category / split
        self.category   = category
        self.augment    = augment_prompts and (split == "train")
        self.transforms = get_transforms(split)
        self.processor  = processor

        self.images = sorted(self.split_dir.glob("*.jpg"))
        self.masks  = {p.stem: p for p in
                       (self.split_dir / "masks").glob("*.png")}

        print(f"  {category}/{split}: {len(self.images)} images loaded")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path  = self.images[idx]
        mask_path = self.masks.get(img_path.stem)

        image = np.array(Image.open(img_path).convert("RGB"))

        if mask_path and mask_path.exists():
            mask = np.array(Image.open(mask_path).convert("L"))
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros(image.shape[:2], dtype=np.float32)

        aug   = self.transforms(image=image, mask=mask)
        image_tensor = aug["image"]                          # [3,H,W]
        #mask_tensor  = torch.from_numpy(aug["mask"]).unsqueeze(0)  # [1,H,W]
        mask_tensor = aug["mask"].clone().detach().unsqueeze(0) \
              if isinstance(aug["mask"], torch.Tensor) \
              else torch.from_numpy(aug["mask"]).unsqueeze(0)
        
        prompt   = random.choice(PROMPT_POOLS[self.category]) \
                   if self.augment else CANONICAL[self.category]
        encoding = self.processor(text=[prompt],
                                  return_tensors="pt",
                                  padding="max_length",
                                  truncation=True)
        return {
            "pixel_values":   image_tensor,
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "mask":           mask_tensor,
            "prompt":         prompt,
            "image_path":     str(img_path),
        }

In [22]:
import json

with open("/kaggle/working/data/taping/valid/_annotations.coco.json") as f:
    data = json.load(f)

# Check first 3 annotations
for ann in data["annotations"][:3]:
    print("keys:", list(ann.keys()))
    print("segmentation:", ann.get("segmentation"))
    print("bbox:", ann.get("bbox"))
    print()

keys: ['id', 'image_id', 'category_id', 'bbox', 'area', 'segmentation', 'iscrowd']
segmentation: []
bbox: [30, 382, 190, 75]

keys: ['id', 'image_id', 'category_id', 'bbox', 'area', 'segmentation', 'iscrowd']
segmentation: []
bbox: [225, 137, 96, 358]

keys: ['id', 'image_id', 'category_id', 'bbox', 'area', 'segmentation', 'iscrowd']
segmentation: []
bbox: [382, 444, 53, 196]



In [23]:
import json
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

def bbox_to_masks(data_dir, category):
    for split in ["train", "valid", "test"]:
        split_dir = Path(data_dir) / category / split
        ann_file  = split_dir / "_annotations.coco.json"

        if not ann_file.exists():
            print(f"Skipping {category}/{split} — no annotation file")
            continue

        mask_dir = split_dir / "masks"
        mask_dir.mkdir(exist_ok=True)

        with open(ann_file) as f:
            data = json.load(f)

        img_info  = {img["id"]: img for img in data["images"]}

        # image_id → list of bboxes [x, y, w, h]
        ann_index = {}
        for ann in data.get("annotations", []):
            iid = ann["image_id"]
            ann_index.setdefault(iid, [])
            if ann.get("bbox"):
                ann_index[iid].append(ann["bbox"])

        count = 0
        for iid, info in img_info.items():
            w, h  = info["width"], info["height"]
            stem  = Path(info["file_name"]).stem
            mask  = Image.new("L", (w, h), 0)
            draw  = ImageDraw.Draw(mask)

            for bbox in ann_index.get(iid, []):
                x, y, bw, bh = bbox
                # Draw filled rectangle for the bbox
                draw.rectangle([x, y, x+bw, y+bh], fill=255)

            mask.save(mask_dir / f"{stem}.png")
            count += 1

        print(f"{category}/{split}: {count} masks regenerated from bboxes")

# Only regenerate taping — crack already has correct polygon masks
bbox_to_masks("/kaggle/working/data", "taping")

taping/train: 936 masks regenerated from bboxes
taping/valid: 250 masks regenerated from bboxes
Skipping taping/test — no annotation file


In [24]:
img_dir  = Path("/kaggle/working/data/taping/valid")
mask_dir = img_dir / "masks"
images   = sorted(img_dir.glob("*.jpg"))[:5]

for img_path in images:
    gt_path = mask_dir / f"{img_path.stem}.png"
    if gt_path.exists():
        gt = np.array(Image.open(gt_path).convert("L"))
        fg = (gt > 0).sum()
        print(f"{img_path.name[:50]}  fg_pixels={fg}")

2000x1500_10_resized_jpg.rf.7a4d808994b9880f703a0d  fg_pixels=58902
2000x1500_14_resized_jpg.rf.281bc0d779e2138d4affdd  fg_pixels=35200
2000x1500_16_resized_jpg.rf.2c8486a43038662573ece3  fg_pixels=56832
2000x1500_17_resized_jpg.rf.6547428c1c6ae5b841d433  fg_pixels=67736
2000x1500_20_resized_jpg.rf.8d96748b59ba7c979a19fe  fg_pixels=131180


In [25]:
MODEL_ID  = "CIDAS/clipseg-rd64-refined"
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Building datasets...")
train_ds = ConcatDataset([
    DrywallDataset(DATA_ROOT, "taping", "train", processor, augment_prompts=True),
    DrywallDataset(DATA_ROOT, "crack",  "train", processor, augment_prompts=True),
])
val_ds = ConcatDataset([
    DrywallDataset(DATA_ROOT, "taping", "valid", processor, augment_prompts=False),
    DrywallDataset(DATA_ROOT, "crack",  "valid", processor, augment_prompts=False),
])

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True,
                          generator=g, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Building datasets...
  taping/train: 936 images loaded
  crack/train: 5164 images loaded
  taping/valid: 250 images loaded
  crack/valid: 201 images loaded

Train batches: 381
Val   batches: 29


In [26]:
class DryWallSegModel(nn.Module):
    def __init__(self, model_id):
        super().__init__()
        self.clipseg = CLIPSegForImageSegmentation.from_pretrained(model_id)
        self.head    = nn.Conv2d(1, 1, kernel_size=1, bias=True)

        # Layer-wise LR: backbone gets 0.1x
        self._backbone_params = list(self.clipseg.clip.parameters())

    def forward(self, pixel_values, input_ids, attention_mask, mask=None):
        out    = self.clipseg(pixel_values=pixel_values,
                              input_ids=input_ids,
                              attention_mask=attention_mask)
        logits = out.logits.unsqueeze(1)   # [B,1,H,W]
        logits = self.head(logits)

        result = {"logits": logits}

        if mask is not None:
            # Resize logits to match mask
            if logits.shape[-2:] != mask.shape[-2:]:
                logits = F.interpolate(logits, size=mask.shape[-2:],
                                       mode="bilinear", align_corners=False)
            bce  = F.binary_cross_entropy_with_logits(logits, mask)
            # Dice
            p    = torch.sigmoid(logits).view(logits.size(0), -1)
            t    = mask.view(mask.size(0), -1)
            dice = 1 - (2*(p*t).sum(1) + 1e-6) / (p.sum(1)+t.sum(1)+1e-6)
            dice = dice.mean()
            loss = 0.5*bce + 0.5*dice
            result.update({"loss": loss, "bce": bce.item(), "dice": dice.item()})

        return result

    def param_groups(self, lr):
        backbone_ids = {id(p) for p in self._backbone_params}
        other = [p for p in self.parameters() if id(p) not in backbone_ids]
        return [
            {"params": self._backbone_params, "lr": lr * 0.1},
            {"params": other,                 "lr": lr},
        ]

model = DryWallSegModel(MODEL_ID).to(DEVICE)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n/1e6:.1f}M")

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

CLIPSegForImageSegmentation LOAD REPORT from: CIDAS/clipseg-rd64-refined
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
clip.text_model.embeddings.position_ids   | UNEXPECTED |  | 
clip.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 150.7M


In [28]:
def compute_metrics(preds, targets, eps=1e-6):
    # preds, targets: float tensors {0,1}
    p = preds.view(preds.size(0), -1)
    t = targets.view(targets.size(0), -1)
    inter = (p * t).sum(1)
    union = p.sum(1) + t.sum(1) - inter
    iou   = (inter + eps) / (union + eps)
    dice  = (2*inter + eps) / (p.sum(1) + t.sum(1) + eps)
    return iou.mean().item(), dice.mean().item()

In [29]:
optimizer = AdamW(model.param_groups(LR), weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler    = torch.amp.GradScaler()

best_miou = 0.0
history   = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────
    model.train()
    t_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Ep{epoch:02d} train", leave=False):
        pv = batch["pixel_values"].to(DEVICE)
        ii = batch["input_ids"].to(DEVICE)
        am = batch["attention_mask"].to(DEVICE)
        mk = batch["mask"].to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda"):
            out  = model(pv, ii, am, mk)
            loss = out["loss"]

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()

    scheduler.step()
    t_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────
    model.eval()
    v_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Ep{epoch:02d} val  ", leave=False):
            pv = batch["pixel_values"].to(DEVICE)
            ii = batch["input_ids"].to(DEVICE)
            am = batch["attention_mask"].to(DEVICE)
            mk = batch["mask"].to(DEVICE)

            with torch.amp.autocast(device_type="cuda"):
                out = model(pv, ii, am, mk)

            v_loss += out["loss"].item()
            preds   = (torch.sigmoid(out["logits"]) >= 0.5).float()
            all_preds.append(preds.cpu())
            all_targets.append(mk.cpu())

    v_loss /= len(val_loader)
    miou, dice = compute_metrics(
        torch.cat(all_preds), torch.cat(all_targets)
    )

    row = {"epoch": epoch, "train_loss": t_loss,
           "val_loss": v_loss, "miou": miou, "dice": dice}
    history.append(row)

    print(f"Ep{epoch:02d} | train={t_loss:.4f} | val={v_loss:.4f} "
          f"| mIoU={miou:.4f} | Dice={dice:.4f}")

    # Save best
    if miou > best_miou:
        best_miou = miou
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "metrics": row},
                   f"{CKPT_DIR}/best_model.pth")
        print(f"  ✓ Best model saved (mIoU={miou:.4f})")

print(f"\nDone. Best mIoU: {best_miou:.4f}")

Ep01 | train=0.4546 | val=0.4563 | mIoU=0.2689 | Dice=0.3711
  ✓ Best model saved (mIoU=0.2689)


Ep02 | train=0.3580 | val=0.4050 | mIoU=0.3657 | Dice=0.5058
  ✓ Best model saved (mIoU=0.3657)


Ep03 | train=0.3326 | val=0.3660 | mIoU=0.4133 | Dice=0.5611
  ✓ Best model saved (mIoU=0.4133)


Ep04 | train=0.3176 | val=0.3414 | mIoU=0.4403 | Dice=0.5901
  ✓ Best model saved (mIoU=0.4403)


Ep05 | train=0.3086 | val=0.3207 | mIoU=0.4610 | Dice=0.6111
  ✓ Best model saved (mIoU=0.4610)


...

Ep26 | train=0.2684 | val=0.2607 | mIoU=0.5243 | Dice=0.6730


Ep27 | train=0.2679 | val=0.2605 | mIoU=0.5246 | Dice=0.6733


Ep28 | train=0.2685 | val=0.2605 | mIoU=0.5245 | Dice=0.6732


Ep29 | train=0.2681 | val=0.2606 | mIoU=0.5245 | Dice=0.6732


Ep30 | train=0.2686 | val=0.2605 | mIoU=0.5246 | Dice=0.6733
  ✓ Best model saved (mIoU=0.5246)

Done. Best mIoU: 0.5246


In [32]:
import os
from pathlib import Path

MASK_OUT = "/kaggle/working/outputs/masks"
os.makedirs(MASK_OUT, exist_ok=True)

# Load best model
ckpt = torch.load(f"{CKPT_DIR}/best_model.pth", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

NORM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1).to(DEVICE)
NORM_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1).to(DEVICE)

def predict_mask(img_path, prompt, threshold=0.5):
    # Load original image and keep original size
    orig_img  = Image.open(img_path).convert("RGB")
    orig_size = (orig_img.height, orig_img.width)

    # Preprocess
    img_resized = orig_img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    arr = torch.from_numpy(np.array(img_resized)).float() / 255.0
    arr = arr.permute(2,0,1).to(DEVICE)
    arr = (arr - NORM_MEAN) / NORM_STD
    pv  = arr.unsqueeze(0)

    # Tokenise prompt
    enc = processor(text=[prompt], return_tensors="pt",
                    padding="max_length", truncation=True)
    ii  = enc["input_ids"].to(DEVICE)
    am  = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda"):
            out    = model(pv, ii, am)
            logits = out["logits"]

    # Upsample to original size
    logits = F.interpolate(logits, size=orig_size,
                           mode="bilinear", align_corners=False)
    mask   = (torch.sigmoid(logits) >= threshold).squeeze().cpu().numpy()
    return (mask * 255).astype(np.uint8), orig_img

# Run on test images — both prompts
test_configs = [
    ("taping", "valid", "segment_taping_area"),
    ("crack",  "valid", "segment_crack"),
]

for category, split, prompt_slug in test_configs:
    img_dir = Path(f"/kaggle/working/data/{category}/{split}")
    prompt  = CANONICAL[category]
    images  = sorted(img_dir.glob("*.jpg"))[:50]  # first 50 per category

    print(f"\nRunning inference: {category}/{split} — '{prompt}'")
    for img_path in tqdm(images):
        mask, _ = predict_mask(img_path, prompt)
        out_name = f"{img_path.stem}__{prompt_slug}.png"
        Image.fromarray(mask, mode="L").save(f"{MASK_OUT}/{out_name}")

print(f"\nMasks saved to {MASK_OUT}")
print(f"Total masks: {len(list(Path(MASK_OUT).glob('*.png')))}")

Loaded best model from epoch 30

Running inference: taping/valid — 'segment taping area'


100%|██████████| 50/50 [00:02<00:00, 20.07it/s]



Running inference: crack/valid — 'segment crack'


100%|██████████| 50/50 [00:02<00:00, 22.23it/s]


Masks saved to /kaggle/working/outputs/masks
Total masks: 100


In [33]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

VIS_OUT = "/kaggle/working/outputs/visuals"
os.makedirs(VIS_OUT, exist_ok=True)

COLORS = {"taping": "#2196F3", "crack": "#F44336"}

def overlay(image, mask, color, alpha=0.45):
    from matplotlib.colors import to_rgb
    r, g, b  = [int(c*255) for c in to_rgb(color)]
    out      = image.copy().astype(float)
    fg       = mask > 0
    out[fg,0] = out[fg,0]*(1-alpha) + r*alpha
    out[fg,1] = out[fg,1]*(1-alpha) + g*alpha
    out[fg,2] = out[fg,2]*(1-alpha) + b*alpha
    return out.astype(np.uint8)

def save_figure(img_path, gt_path, prompt, category, out_path):
    image    = np.array(Image.open(img_path).convert("RGB"))
    gt_mask  = np.array(Image.open(gt_path).convert("L")) if gt_path and Path(gt_path).exists() \
               else np.zeros(image.shape[:2], dtype=np.uint8)
    pred_mask, _ = predict_mask(img_path, prompt)

    # Resize pred to match original
    color = COLORS[category]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("#1a1a1a")

    for ax, title, img in zip(axes,
        ["Original", "Ground Truth", "Prediction"],
        [image, overlay(image, gt_mask, color), overlay(image, pred_mask, color)]):
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=13, fontweight="bold")
        ax.axis("off")

    # Compute IoU
    p = pred_mask > 0
    t = gt_mask   > 0
    iou = (p & t).sum() / ((p | t).sum() + 1e-6)
    fig.suptitle(f'"{prompt}"   |   IoU: {iou:.3f}',
                 color="white", fontsize=11, y=0.02)

    plt.tight_layout(rect=[0,0.06,1,1])
    plt.savefig(out_path, dpi=120, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {out_path.name}  [IoU={iou:.3f}]")

# Generate 4 examples per category
for category in ["taping", "crack"]:
    prompt   = CANONICAL[category]
    img_dir  = Path(f"/kaggle/working/data/{category}/valid")
    mask_dir = img_dir / "masks"
    images   = sorted(img_dir.glob("*.jpg"))[:4]

    print(f"\n{category}:")
    for i, img_path in enumerate(images):
        gt_path  = mask_dir / f"{img_path.stem}.png"
        out_path = Path(VIS_OUT) / f"{category}_example_{i+1:02d}.png"
        save_figure(img_path, gt_path, prompt, category, out_path)

print(f"\nAll visuals saved to {VIS_OUT}")


taping:
  Saved: taping_example_01.png  [IoU=0.699]
  Saved: taping_example_02.png  [IoU=0.673]
  Saved: taping_example_03.png  [IoU=0.721]
  Saved: taping_example_04.png  [IoU=0.541]

crack:
  Saved: crack_example_01.png  [IoU=0.799]
  Saved: crack_example_02.png  [IoU=0.502]
  Saved: crack_example_03.png  [IoU=0.518]
  Saved: crack_example_04.png  [IoU=0.335]

All visuals saved to /kaggle/working/outputs/visuals


In [34]:
from pathlib import Path
import numpy as np
from PIL import Image

img_dir  = Path("/kaggle/working/data/taping/valid")
mask_dir = img_dir / "masks"
images   = sorted(img_dir.glob("*.jpg"))[:10]

for img_path in images:
    gt_path = mask_dir / f"{img_path.stem}.png"
    if gt_path.exists():
        gt = np.array(Image.open(gt_path).convert("L"))
        fg = (gt > 0).sum()
        print(f"{img_path.name[:50]}  fg_pixels={fg}")
    else:
        print(f"{img_path.name[:50]}  NO MASK")

2000x1500_10_resized_jpg.rf.7a4d808994b9880f703a0d  fg_pixels=58902
2000x1500_14_resized_jpg.rf.281bc0d779e2138d4affdd  fg_pixels=35200
2000x1500_16_resized_jpg.rf.2c8486a43038662573ece3  fg_pixels=56832
2000x1500_17_resized_jpg.rf.6547428c1c6ae5b841d433  fg_pixels=67736
2000x1500_20_resized_jpg.rf.8d96748b59ba7c979a19fe  fg_pixels=131180
2000x1500_21_resized_jpg.rf.a260d6194a3531c4ac73dd  fg_pixels=65423
2000x1500_22_resized_jpg.rf.842f265d1960fdff3690ce  fg_pixels=73554
2000x1500_23_resized_jpg.rf.2b975410256245b5fb430d  fg_pixels=37004
2000x1500_26_resized_jpg.rf.013a58baeb8f5fda4105a7  fg_pixels=154088
2000x1500_27_resized_jpg.rf.031ab0f4e6dcb59da72e3f  fg_pixels=134291


In [35]:
print("── Per-prompt metrics on validation set ──────────────────")
print(f"{'Prompt':<30} {'mIoU':>8} {'Dice':>8}")
print("-" * 50)

for category in ["taping", "crack"]:
    prompt   = CANONICAL[category]
    img_dir  = Path(f"/kaggle/working/data/{category}/valid")
    mask_dir = img_dir / "masks"
    images   = sorted(img_dir.glob("*.jpg"))

    all_iou, all_dice = [], []
    for img_path in tqdm(images, desc=category, leave=False):
        gt_path = mask_dir / f"{img_path.stem}.png"
        if not gt_path.exists():
            continue
        pred, _ = predict_mask(img_path, prompt)
        gt      = np.array(Image.open(gt_path).convert("L"))

        p = (pred > 0).astype(float).flatten()
        t = (gt   > 0).astype(float).flatten()
        inter = (p * t).sum()
        union = p.sum() + t.sum() - inter
        iou   = (inter + 1e-6) / (union + 1e-6)
        dice  = (2*inter + 1e-6) / (p.sum() + t.sum() + 1e-6)
        all_iou.append(iou)
        all_dice.append(dice)

    print(f"{prompt:<30} {np.mean(all_iou):>8.4f} {np.mean(all_dice):>8.4f}")

print("-" * 50)

── Per-prompt metrics on validation set ──────────────────
Prompt                             mIoU     Dice
--------------------------------------------------


segment taping area              0.5344   0.6852


segment crack                    0.5141   0.6596
--------------------------------------------------


In [ ]:
import os, random, json, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, CLIPSegForImageSegmentation
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ── Constants ──────────────────────────────────────────────────
SEED       = 42
DEVICE     = torch.device("cuda")
IMG_SIZE   = 352
CKPT_DIR   = "/kaggle/working/models"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

NORM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1).to(DEVICE)
NORM_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1).to(DEVICE)

CANONICAL = {
    "taping": "segment taping area",
    "crack":  "segment crack",
}

MODEL_ID  = "CIDAS/clipseg-rd64-refined"
processor = AutoProcessor.from_pretrained(MODEL_ID)

# ── Model definition ───────────────────────────────────────────
class DryWallSegModel(nn.Module):
    def __init__(self, model_id):
        super().__init__()
        self.clipseg = CLIPSegForImageSegmentation.from_pretrained(model_id)
        self.head    = nn.Conv2d(1, 1, kernel_size=1, bias=True)
        self._backbone_params = list(self.clipseg.clip.parameters())

    def forward(self, pixel_values, input_ids, attention_mask, mask=None):
        out    = self.clipseg(pixel_values=pixel_values,
                              input_ids=input_ids,
                              attention_mask=attention_mask)
        logits = out.logits.unsqueeze(1)
        logits = self.head(logits)
        result = {"logits": logits}
        if mask is not None:
            if logits.shape[-2:] != mask.shape[-2:]:
                logits = F.interpolate(logits, size=mask.shape[-2:],
                                       mode="bilinear", align_corners=False)
            bce  = F.binary_cross_entropy_with_logits(logits, mask)
            p    = torch.sigmoid(logits).view(logits.size(0), -1)
            t    = mask.view(mask.size(0), -1)
            dice = 1 - (2*(p*t).sum(1)+1e-6) / (p.sum(1)+t.sum(1)+1e-6)
            loss = 0.5*bce + 0.5*dice.mean()
            result.update({"loss": loss, "bce": bce.item(), "dice": dice.mean().item()})
        return result

# ── Load best checkpoint ───────────────────────────────────────
model = DryWallSegModel(MODEL_ID).to(DEVICE)
ckpt  = torch.load(f"{CKPT_DIR}/best_model.pth", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Model loaded from epoch {ckpt['epoch']}")
print(f"Val mIoU: {ckpt['metrics']['miou']:.4f}")
print(f"Val Dice: {ckpt['metrics']['dice']:.4f}")
print(f"Device  : {DEVICE} ({torch.cuda.get_device_name(0)})")

In [36]:
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import io
import torch
import torch.nn.functional as F
from pathlib import Path

# ── predict_mask defined here ──────────────────────────────────
NORM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1).to(DEVICE)
NORM_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1).to(DEVICE)

def predict_mask(img_input, prompt, threshold=0.5):
    if isinstance(img_input, (str, Path)):
        orig_img = Image.open(img_input).convert("RGB")
    else:
        orig_img = img_input.convert("RGB")

    orig_size   = (orig_img.height, orig_img.width)
    img_resized = orig_img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    arr = torch.from_numpy(np.array(img_resized)).float() / 255.0
    arr = arr.permute(2,0,1).to(DEVICE)
    arr = (arr - NORM_MEAN) / NORM_STD
    pv  = arr.unsqueeze(0)

    enc = processor(text=[prompt], return_tensors="pt",
                    padding="max_length", truncation=True)
    ii  = enc["input_ids"].to(DEVICE)
    am  = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda"):
            out    = model(pv, ii, am)
            logits = out["logits"]

    logits = F.interpolate(logits, size=orig_size,
                           mode="bilinear", align_corners=False)
    mask   = (torch.sigmoid(logits) >= threshold).squeeze().cpu().numpy()
    return (mask * 255).astype(np.uint8), orig_img

# ── Widget ─────────────────────────────────────────────────────
upload       = widgets.FileUpload(accept="image/*", multiple=False)
prompt_input = widgets.Text(
    value="segment crack",
    description="Prompt:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)
button = widgets.Button(description="Run Segmentation",
                        button_style="primary",
                        layout=widgets.Layout(width="200px"))
output = widgets.Output()

def on_click(b):
    output.clear_output()
    with output:
        if not upload.value:
            print("Please upload an image first.")
            return

        uploaded_file = upload.value[0]
        img    = Image.open(io.BytesIO(uploaded_file["content"])).convert("RGB")
        prompt = prompt_input.value.strip()

        print(f"Image size : {img.width}×{img.height}")
        print(f"Prompt     : '{prompt}'")
        print("Running...")

        mask, _ = predict_mask(img, prompt)
        fg_pct  = (mask > 0).mean() * 100

        from matplotlib.colors import to_rgb
        COLOR   = "#F44336" if "crack" in prompt else "#2196F3"
        r, g, b = [int(c*255) for c in to_rgb(COLOR)]

        img_arr = np.array(img)
        overlay = img_arr.copy().astype(float)
        fg = mask > 0
        overlay[fg, 0] = overlay[fg, 0]*0.55 + r*0.45
        overlay[fg, 1] = overlay[fg, 1]*0.55 + g*0.45
        overlay[fg, 2] = overlay[fg, 2]*0.55 + b*0.45
        overlay = overlay.astype(np.uint8)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.patch.set_facecolor("#1a1a1a")

        for ax, title, im in zip(
            axes,
            ["Original", f'"{prompt}"', "Mask only"],
            [img_arr, overlay, mask]
        ):
            ax.imshow(im, cmap="gray" if title == "Mask only" else None)
            ax.set_title(title, color="white", fontsize=12, fontweight="bold")
            ax.axis("off")

        plt.suptitle(f"Foreground detected: {fg_pct:.1f}% of image",
                     color="white", fontsize=11, y=0.02)
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        plt.show()

        out_slug = prompt.strip().lower().replace(" ", "_")
        out_path = f"/kaggle/working/test_output__{out_slug}.png"
        Image.fromarray(mask, mode="L").save(out_path)
        print(f"Mask saved → {out_path}")

button.on_click(on_click)

print("Step 1: Upload an image")
print("Step 2: Type your prompt  (e.g. 'segment crack' or 'segment taping area')")
print("Step 3: Click Run Segmentation")
print()
display(upload, prompt_input, button, output)

Step 1: Upload an image
Step 2: Type your prompt  (e.g. 'segment crack' or 'segment taping area')
Step 3: Click Run Segmentation



FileUpload(value=(), accept='image/*', description='Upload')

Text(value='segment crack', description='Prompt:', layout=Layout(width='400px'), style=TextStyle(description_w…

Button(button_style='primary', description='Run Segmentation', layout=Layout(width='200px'), style=ButtonStyle…

Output()

In [40]:
import shutil
from pathlib import Path

# Zip the entire working directory outputs
shutil.make_archive(
    "/kaggle/working/drywall_qa_final",
    "zip",
    "/kaggle/working"
)
print("Zipped!")

# Show size
size = Path("/kaggle/working/drywall_qa_final.zip").stat().st_size / (1024*1024)
print(f"Size: {size:.1f} MB")

Zipped!
Size: 703.5 MB


In [44]:
import shutil
from pathlib import Path

# Copy model to /kaggle/working/ root — this is what Kaggle exposes for download
shutil.copy(
    "/kaggle/working/models/best_model.pth",
    "/kaggle/working/best_model.pth"
)
print("Done — file is ready")
print(f"Size: {Path('/kaggle/working/best_model.pth').stat().st_size / 1024**2:.1f} MB")

Done — file is ready
Size: 575.2 MB


In [ ]:
from IPython.display import Javascript

JS = """
var link = document.createElement('a');
link.href = '/kaggle/working/best_model.pth';
link.download = 'best_model.pth';
document.body.appendChild(link);
link.click();
document.body.removeChild(link);
"""
#Javascript(JS)
# ```

# Or just go directly to this URL in your browser (replace `USERNAME` and `NOTEBOOK-NAME` with yours):
# ```
# https://www.kaggle.com/code/USERNAME/NOTEBOOK-NAME/output